# QAOA angle prediction — смесь угловых экспертов

**Задача.** Для каждого вектора линейных коэффициентов `h` (12 кубитов, фиксированная `J`)
предсказать 5 углов `gamma` и 5 углов `beta` схемы QAOA глубины 5. Метрика —
`P(ground)`, усреднённая по 500 инстансам.

**Как запускать.** `Runtime → Run all`. Ноутбук ничего не требует править руками:
он клонирует публичный репозиторий, берёт оттуда `J.npy`, `h_test.npy` и неизменённый
`QAOA.py` организаторов, загружает обученную модель и пишет `submission.csv`.
GPU не обязателен, но с ним инференс быстрее.

**Лимит.** Инференс на всём `h_test` укладывается в 10 минут — ячейка с инференсом
печатает своё время явно, чтобы это можно было проверить, а не принимать на веру.

**Что внутри модели.** Обучаемый *словарь* угловых векторов плюс *гейт*, который
по `h` выбирает из него. Обучается целиком через дифференцируемый симулятор
организаторов, на той самой метрике, которой оценивают. Размеченных углов нигде нет.
Почему именно так — в разделе «Форма задачи» ниже; это не архитектурный вкус,
а следствие измерений.

## 1. Установка

Клонируем репозиторий с кодом и данными. В Colab уже есть `torch` и `numpy`, больше ничего не нужно.

In [ ]:
import os, sys, subprocess, pathlib

REPO_URL = "https://github.com/brkdrd/sberchall"
REPO = pathlib.Path("/content/sberchall") if pathlib.Path("/content").exists() else pathlib.Path("sberchall")

if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=False)

os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import numpy as np, torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} | device {DEVICE}"
      + (f" | {torch.cuda.get_device_name(0)}" if DEVICE == "cuda" else ""))

## 2. Данные и симулятор

`src/qaoa_ref.py` — это файл `QAOA.py` организаторов, **побайтово неизменённый**
(проверяем это прямо здесь). Ничего нашего между углами и оценкой не стоит:
все числа ниже получены через `QAOA.p_ground`.

In [ ]:
import filecmp
from src.qaoa_ref import QAOA, P as DEPTH

DATA = pathlib.Path("data/raw/sberdata")
same = open(DATA / "QAOA.py", "rb").read().replace(b"\r\n", b"\n") == \
       open("src/qaoa_ref.py", "rb").read().replace(b"\r\n", b"\n")
print("src/qaoa_ref.py идентичен QAOA.py организаторов:", same)
assert same, "симулятор разошёлся с оригиналом"

J       = np.load(DATA / "J.npy")
h_train = np.load(DATA / "h_train.npy")
h_test  = np.load(DATA / "h_test.npy")
sim = QAOA(J, device=DEVICE)
H_TRAIN = torch.tensor(h_train, dtype=torch.float32, device=DEVICE)
H_TEST  = torch.tensor(h_test,  dtype=torch.float32, device=DEVICE)
print(f"J {J.shape} | h_train {h_train.shape} | h_test {h_test.shape} | глубина p={DEPTH}")

## 3. Форма задачи

Три измеренных факта, из которых следует архитектура. Все три проверяются ячейкой ниже
или ссылаются на эксперименты в репозитории (`src/gscan.py`, `src/library.py`, `src/hard.py`).

**(а) `J` — матрица ранга 1.** `J = v vᵀ − diag(v²)`, где `v_i = cos(π k_i / 5.5)`,
`k = [0,3,5,2,1,4,4,1,2,5,3,0]`. Ошибка восстановления ровно 0.0. Отсюда
`E(z) = ½(v·z)² − ½‖v‖² + h·z`, то есть вся связность задачи — это один коллективный
член `M² `, где `M = Σ v_i z_i`.

**(б) У метрики есть точная группа симметрий из 128 элементов.** `v_i = v_{11−i}`, поэтому
перестановка кубитов внутри любой из шести пар `(i, 11−i)` не меняет `J` (2⁶ = 64), и
`h → −h` тоже ничего не меняет (`X^⊗12` коммутирует с миксером). Все 128 оставляют
оптимальные углы теми же. Подавать сети сырое `h` — значит заставлять её выучить 128 копий
одной функции; мы складываем эту группу точно, признаками.

**(в) Задача — это выбор, а не регрессия и не поиск.** 500 выигравших угловых векторов,
скрещенные со всеми 500 инстансами, дают средний `P` = 0.315 **вообще без поиска**, против
0.312 у 16-тысячестартового поиска, который их и породил. Хорошие бассейны общие; с `h`
меняется только то, *какой* из них правильный. Регрессия здесь ломается, потому что при
фиксированном `h` хорошие углы образуют несколько разделённых сгустков, и L2-приближение
садится в их среднее — точку, не лежащую ни в одном бассейне.

In [ ]:
# (а) J ранга 1 — проверка
k = np.array([0, 3, 5, 2, 1, 4, 4, 1, 2, 5, 3, 0])
v = np.cos(np.pi * k / 5.5)
print("ошибка восстановления J:", np.abs(np.outer(v, v) - np.diag(v**2) - J).max())

# (б) P(ground) инвариантна ко всем 128 симметриям — проверка на реальных углах
ang = torch.rand(8, 2 * DEPTH, device=DEVICE) * 2 - 1
p0 = sim.p_ground(H_TRAIN[:8], ang[:, :DEPTH], ang[:, DEPTH:])
worst = 0.0
for mask in range(64):
    hp = H_TRAIN[:8].clone()
    for j in range(6):
        if mask >> j & 1:
            hp[:, [j, 11 - j]] = hp[:, [11 - j, j]]
    for s in (1.0, -1.0):
        p = sim.p_ground(s * hp, ang[:, :DEPTH], ang[:, DEPTH:])
        worst = max(worst, (p - p0).abs().max().item())
print("максимальное отклонение P(ground) по всей орбите из 128 элементов:", f"{worst:.2e}")

## 4. Признаки: группа симметрий складывается точно

`canonical_features` приводит `h` к единственному представителю его орбиты — сначала
фиксирует глобальный знак, потом сортирует внутри пар (порядок важен: отрицание `h`
меняет местами min и max внутри каждой пары, так что эти два шага не коммутируют).
Ключ знака — `Σ h_i v_i`: он инвариантен к перестановкам, нечётен к `h → −h` и непрерывен.
Намагниченность основного состояния на эту роль не годится — она дискретна и ровно нулевая
на 43 из 500 инстансов `h_train`.

К этому добавляются величины, которые метрика и так вычисляет полным перебором 4096
конфигураций: основное состояние `z*`, щель `E₁ − E₀`, `E₀` и `min |h_i|`.

Отклонение признаков по всей орбите из 128 элементов — **ровно 0.0**, это проверяется ниже.

In [ ]:
from src.moe import canonical_features, FEATURE_DIM

f0 = canonical_features(sim, H_TEST)
worst = 0.0
for mask in range(64):
    hp = H_TEST.clone()
    for j in range(6):
        if mask >> j & 1:
            hp[:, [j, 11 - j]] = hp[:, [11 - j, j]]
    for s in (1.0, -1.0):
        worst = max(worst, (canonical_features(sim, s * hp) - f0).abs().max().item())
print(f"признаков: {FEATURE_DIM}")
print(f"максимальное отклонение признаков по орбите из 128 элементов: {worst:.3e}")
print(f"различных канонических строк среди 500 инстансов h_test: "
      f"{len({tuple(r) for r in f0.round(decimals=5).tolist()})}")

## 5. Модель

**Словарь** `C ∈ R^{M×10}` — M угловых векторов, обучаемых. `gamma` зажата в
`GAMMA_BOX·tanh(·)`: замер по гамма-лестнице (`src/gscan.py`) показал полезную область
`|gamma| ~ 0.1..1` и монотонный развал выше 1.0. `beta` не зажимаем — миксер π-периодичен,
убегать некуда.

**Гейт** — MLP из канонических признаков в логиты над словарём.

**Функция потерь** — вероятность смеси:

$$\mathcal{L}(h) = -\log \sum_m \mathrm{softmax}(\mathrm{gate}(h))_m \cdot P_{\mathrm{ground}}(h, C_m)$$

Максимизируется именно смесь, а не лучший эксперт: тогда каждый эксперт получает градиент
пропорционально ответственности, которую ему назначил гейт. Эксперты специализируются,
гейт разбивает пространство — это EM, выполненный градиентным спуском сквозь квантовую схему.

In [ ]:
from src.moe import AngleMoE

model = AngleMoE(n_experts=256, d_model=256, gamma_box=1.6).to(DEVICE)
n_par = sum(p.numel() for p in model.parameters())
print(f"экспертов {model.n_experts} | параметров {n_par:,} "
      f"(словарь {model.codebook.numel():,}, гейт {n_par - model.codebook.numel():,})")

## 6. Обучение

Обучающие инстансы бесплатны: `h_train` — это i.i.d. `U(−1, 1)` (тест Колмогорова—Смирнова,
p = 0.77), поэтому свежие `h` синтезируются на каждом шаге, а официальный `h_train`
**не обучается вообще** и служит только валидацией.

По умолчанию ниже стоит короткий прогон, чтобы `Run all` завершился за разумное время и
было видно, что кривая обучения идёт вверх. Для полного обучения поставьте
`TRAIN_FULL = True` — это примерно 4000 итераций (порядка часа на T4, ~15 минут на
современной карте). Правила это разрешают: обучение может не укладываться в лимиты Colab.

Готовый чекпойнт лежит в репозитории (`models/moe_best.pt`), и раздел инференса возьмёт
именно его, так что для воспроизведения сабмита обучение запускать не обязательно.

In [ ]:
from src.moe import train, evaluate

TRAIN_FULL = False          # ← поставьте True для полного обучения

iters, batch, active = (4000, 64, 64) if TRAIN_FULL else (150, 32, 32)
opt = torch.optim.Adam(model.parameters(), lr=3e-4)
hist = train(sim, model, opt, iters=iters, batch=batch, active=active,
             device=DEVICE, h_val=H_TRAIN, log_every=max(1, iters // 10),
             out_dir="runs/moe_notebook", seed=0)
print(f"\nвалидация (гейт без полировки): {hist[0]['val_gate_P']:.5f} "
      f"→ {hist[-1]['val_gate_P']:.5f}")

## 7. Инференс на `h_test` → `submission.csv`

Гейт оценивает все M экспертов против `h` (только прямые проходы), лучшие `K` полируются
Adam'ом сквозь симулятор, победитель выбирается **после** полировки — Adam уводит каждого
кандидата в оптимум его собственного бассейна, поэтому лучший старт не есть лучший финиш.

Стоимость: `M + 2·K·steps` прямых проходов на инстанс. Ячейка печатает своё время —
лимит 600 секунд.

In [ ]:
import time
from src.moe import load, write_submission

CKPT = pathlib.Path("models/moe_best.pt")
if CKPT.exists():
    infer_model = load(CKPT, DEVICE)
    print(f"загружен чекпойнт {CKPT}")
else:
    infer_model = model
    print("чекпойнта в репозитории нет — используется модель, обученная выше")

t0 = time.time()
res = evaluate(sim, infer_model, H_TEST, top_k=8, polish_steps=300,
               lr_gamma=0.05, lr_beta=0.03, chunk=2048)
dt = time.time() - t0

write_submission("submission.csv", res["angles"])
print(f"h_test: гейт без полировки {res['gate_P']:.5f} | после полировки {res['polished_P']:.5f}")
print(f"инференс на 500 инстансах: {dt:.0f} с (лимит 600 с) — {'OK' if dt < 600 else 'ПРЕВЫШЕН'}")
print("записан submission.csv")

## 8. Проверка: тот же путь на `h_train`

`h_test` без ответов, поэтому число, которое можно сверить, — это метрика на `h_train`,
полученная **тем же самым кодом**. Углы для `h_train` модель видит впервые: обучение шло
на синтетических `h`.

In [ ]:
t0 = time.time()
val = evaluate(sim, infer_model, H_TRAIN, top_k=8, polish_steps=300,
               lr_gamma=0.05, lr_beta=0.03, chunk=2048)
print(f"h_train: гейт без полировки {val['gate_P']:.5f} | "
      f"после полировки {val['polished_P']:.5f}  [{time.time() - t0:.0f} с]")

p = val["per_instance"].cpu().numpy()
print(f"по инстансам: медиана {np.median(p):.5f} | min {p.min():.5f} | max {p.max():.5f}")
print(f"инстансов с P > 0.5: {(p > 0.5).sum()} из {len(p)}")

## 9. Формат сабмита

500 строк, колонки `id, gamma_0..gamma_4, beta_0..beta_4`. Проверяем явно, что файл
соответствует формату и что углы **зависят от `h`** — сдача константных углов запрещена
правилами.

In [ ]:
import csv
rows = list(csv.reader(open("submission.csv")))
head, body = rows[0], rows[1:]
a = np.array([[float(x) for x in r[1:]] for r in body])
print(f"колонки: {head}")
print(f"строк: {len(body)} | углов в строке: {a.shape[1]}")
print(f"уникальных угловых векторов: {len(np.unique(a.round(6), axis=0))} из {len(a)}")
print(f"id: {body[0][0]}..{body[-1][0]} | все конечны: {np.isfinite(a).all()}")
assert len(body) == 500 and a.shape[1] == 10 and np.isfinite(a).all()
assert len(np.unique(a.round(6), axis=0)) > 1, "константные углы запрещены правилами"
print("\nформат в порядке")